# Neural Net Inference in 30 Lines of K

**What if you could run a neural network in a language where the entire forward pass fits in a tweet?**

[K](https://en.wikipedia.org/wiki/K_(programming_language)) is an array programming language created by Arthur Whitney. It's famous for expressing complex numerical algorithms in extraordinarily dense notation. [Kona](https://github.com/kevinlawler/kona) is an open-source implementation of K3.

In this cookbook, we use Claude to:
1. **Generate** a complete 2-layer neural network in K (training + inference)
2. **Explain** each line of the resulting ~30-line program
3. **Validate** the output by actually running it in the Kona interpreter

This demonstrates Claude's ability to work with extremely terse, specialized languages — not just mainstream Python/JS — and produce correct, runnable code in domains where training data is sparse.

### Why this matters

- K has virtually no presence in LLM training corpora compared to Python
- The notation is so dense that a single misplaced character breaks everything
- Array thinking (no loops, everything is implicit map/reduce) requires genuine understanding
- This is a real stress test of Claude's code generation beyond comfort-zone languages

## Setup

You need:
- An Anthropic API key
- [Kona](https://github.com/kevinlawler/kona) installed (`k` binary in PATH) — optional, only needed to run the generated code

```bash
# Install Kona (Linux/macOS)
git clone https://github.com/kevinlawler/kona.git && cd kona && make && sudo cp k /usr/local/bin/

# Or on Termux (Android)
pkg install kona
```

In [ ]:
import anthropic
import subprocess
import shutil
import tempfile
import os
import re

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

## The K reference card we give Claude

K is niche enough that we need to remind Claude of the key idioms. This reference card covers the patterns needed for a neural net: dot products via `_dot`, broadcasting via eachleft (`\:`), outer products via `*\:`, and the adverb system.

In [ ]:
K_REFERENCE = """
Kona (K3) Quick Reference for Neural Nets:

CORE VERBS: + - * % (divide) | (max/reverse) & (min/where) ^ (power)
  ! (mod/enumerate) < > = ~ @ ? _ (floor/drop) , (join) # (count/take) $ (format)
FOLD/SCAN: +/ (sum) */ (product) |/ (max) &/ (min)  +\ (running sum)
ADVERBS: / (over) \ (scan) ' (each) /: (eachright) \: (eachleft)
BUILTINS: _exp _log _sqrt _tanh _dot (dot product) _mul (matrix multiply)
LAMBDAS: {x+y} — implicit args x,y,z.  {[a;b] a+b} — named args.
ASSIGNMENT: a:1 (local)  a::1 (global, required inside functions)
CONTROL: do[n;e1;...;en]  while[cond;e1;...;en]  if[cond;e1;...;en]
RANDOM: n _draw 0 gives n random floats in [0,1). n _draw m gives n random ints in [0,m).
I/O: `0: "text" prints to stdout.  5: x formats x as string.
COMMENTS: / at start of line, or space before /

KEY PATTERNS FOR NEURAL NETS:
  W _dot\: v           / dot product of each row of W with vector v
  (+W) _dot\: v        / dot with transposed W (for backprop)
  grad*\:input         / outer product (for weight gradient)
  sig:{1.0%(1.0+_exp(-x))}  / sigmoid
  dsig:{x*(1.0-x)}          / sigmoid derivative (on activated value)

GOTCHAS:
  % is DIVIDE not mod. ! dyadic is mod.
  Evaluation is right-to-left: 2*3+1 = 2*4 = 8, not 7.
  _draw 0 returns floats [0,1). Use ((n _draw 0)*2.0)-1.0 for [-1,1).
  Global assignment :: required inside functions or updates won't persist.
"""

## Step 1: Ask Claude to write a neural network in K

We ask for a complete XOR solver: 2 inputs, 4 hidden units (sigmoid), 1 output, trained with SGD.

In [ ]:
generation_prompt = f"""You are an expert K/Kona (K3) programmer. Write a COMPLETE, RUNNABLE Kona script that:

1. Defines a 2-layer neural net: 2 inputs -> 4 hidden (sigmoid) -> 1 output (sigmoid)
2. Trains on XOR: inputs (0,0),(0,1),(1,0),(1,1) -> targets 0,1,1,0
3. Uses SGD with learning rate 2.0, trains for 20000 epochs
4. After training, prints predictions for all 4 inputs

Requirements:
- Use _dot\: for "dot product of each row of weight matrix with a vector"
- Use *\: for outer products (weight gradients)
- Use (+W) for transpose in backprop
- Targets Y should be 1-element vectors: (,0.0;,1.0;,1.0;,0.0)
- Init weights with ((n _draw 0)*2.0)-1.0 for range [-1,1)
- Use \\r seed to set random seed for reproducibility
- End script with \\\\ to exit
- Keep it under 35 lines total

{K_REFERENCE}

Output ONLY the K code inside a ```k code block. No explanation."""

print("Asking Claude to write a neural net in K...\n")
response = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    messages=[{{"role": "user", "content": generation_prompt}}]
)

raw_response = response.content[0].text
print(raw_response)

## Step 2: Extract the K code

In [ ]:
match = re.search(r"```k\n(.*?)```", raw_response, re.DOTALL)
if not match:
    match = re.search(r"```\n(.*?)```", raw_response, re.DOTALL)

k_code = match.group(1).strip() if match else raw_response.strip()

print(f"Generated K program: {len(k_code.splitlines())} lines\n")
for i, line in enumerate(k_code.splitlines(), 1):
    print(f"{i:3d} | {line}")

## Step 3: Ask Claude to explain the code line by line

K code is notoriously dense. Let's have Claude break it down for humans.

In [ ]:
explain_prompt = f"""Here is a neural network written in Kona (K3). Explain each line concisely.
For each line, explain: what K idiom is used, and what it does in neural net terms.
Keep each explanation to 1 sentence. Skip comment-only lines.

```k
{k_code}
```"""

explanation = client.messages.create(
    model=MODEL,
    max_tokens=4096,
    messages=[{"role": "user", "content": explain_prompt}]
)

print(explanation.content[0].text)

## Step 4: Run it in Kona

If Kona is installed, we execute Claude's code and check the predictions.

**Expected:** After training, predictions should approach:
- (0,0) -> ~0.0
- (0,1) -> ~1.0  
- (1,0) -> ~1.0
- (1,1) -> ~0.0

In [ ]:
def run_k(code, timeout=60):
    """Run K code in Kona and return (stdout, stderr, returncode)."""
    kona = shutil.which("k")
    if not kona:
        return None, "Kona not installed", -1
    with tempfile.NamedTemporaryFile(mode="w", suffix=".k", delete=False) as f:
        # Append \\ so the interpreter exits after running
        f.write(code + "\n\\\\\n")
        path = f.name
    try:
        r = subprocess.run([kona, path], capture_output=True, text=True, timeout=timeout)
        return r.stdout, r.stderr, r.returncode
    finally:
        os.unlink(path)


kona_path = shutil.which("k")
if kona_path:
    print(f"Kona found at: {kona_path}")
    stdout, stderr, rc = run_k(k_code)
    if stdout:
        print(f"\nOUTPUT:\n{stdout}")
    if stderr:
        print(f"\nERROR:\n{stderr[:500]}")
else:
    print("Kona not found. Save the code as xor.k and run: k xor.k")

## Step 5: Self-repair

Array languages are unforgiving — a single wrong character crashes everything. If Claude's code errored, we feed the error back and let it fix itself. This loop converges in 1-2 iterations.

In [ ]:
def self_repair(code, max_attempts=3):
    """Let Claude iteratively fix K code until it runs."""
    for attempt in range(max_attempts):
        stdout, stderr, rc = run_k(code)
        if stdout is None:
            print("Kona not available, skipping self-repair.")
            return code

        if rc == 0 and stderr.strip() == "":
            print(f"Code runs clean on attempt {attempt + 1}!")
            if stdout.strip():
                print(f"Output:\n{stdout.strip()}")
            return code

        print(f"\nAttempt {attempt + 1} failed: {stderr[:300]}")
        print("Asking Claude to fix...")

        fix_prompt = f"""This Kona (K3) script has an error. Fix it and return the corrected code.

CODE:
```k
{code}
```

ERROR:
```
{stderr[:500]}
```

{K_REFERENCE}

IMPORTANT: Use _dot\: for row-wise dot products. Use *\: for outer products.
Output ONLY the fixed code in a ```k block. Under 35 lines."""

        fix_response = client.messages.create(
            model=MODEL,
            max_tokens=2048,
            messages=[{"role": "user", "content": fix_prompt}]
        )
        fix_text = fix_response.content[0].text
        m = re.search(r"```k\n(.*?)```", fix_text, re.DOTALL)
        if not m:
            m = re.search(r"```\n(.*?)```", fix_text, re.DOTALL)
        if m:
            code = m.group(1).strip()
            print(f"Got fixed version ({len(code.splitlines())} lines)")
        else:
            print("Could not extract fixed code.")
            break

    return code


if shutil.which("k"):
    final_code = self_repair(k_code)
else:
    final_code = k_code
    print("Kona not installed — skipping self-repair loop.")

## Bonus: Hand-tested reference implementation

Here's a known-good XOR neural net in K, tested on Kona. This uses the same `_dot\:` and `*\:` patterns. Compare it to Claude's version above.

The entire thing — init, forward pass, backprop, 20000-epoch training loop, and inference — is **28 lines**.

In [ ]:
reference_k = """\
/ XOR neural net in Kona - 2-4-1 sigmoid SGD
sig:{1.0%(1.0+_exp(-x))}
dsig:{x*(1.0-x)}

X:(0 0;0 1;1 0;1 1)*1.0
Y:(,0.0;,1.0;,1.0;,0.0)

\\r 42
W1:(2 4)#((8 _draw 0)*2.0)-1.0
b1:((4 _draw 0)*2.0)-1.0
W2:(4 1)#((4 _draw 0)*2.0)-1.0
b2:((1 _draw 0)*2.0)-1.0
lr:2.0

fwd:{[xi]
  h:sig (W1 _dot\\: xi)+b1
  o:sig (W2 _dot\\: h)+b2
  (h;o)}

step:{[xi;yi]
  r:fwd xi
  h:r 0; o:r 1
  eo:o-yi
  od:eo*dsig o
  dh:((+W2) _dot\\: od)*dsig h
  W2::W2-(lr*(od*\\:h))
  b2::b2-(lr*od)
  W1::W1-(lr*(dh*\\:xi))
  b1::b1-(lr*dh)
  0}

do[20000; step[X 0;Y 0]; step[X 1;Y 1]; step[X 2;Y 2]; step[X 3;Y 3]]

`0: "XOR predictions:\\n"
i:0
while[i<4
  r:fwd X i
  `0: (5: X i)," -> ",(5: r 1),"\\n"
  i:i+1]
"""

print(f"Reference: {len(reference_k.strip().splitlines())} lines\n")

stdout, stderr, rc = run_k(reference_k)
if stdout:
    print(stdout)
if stderr:
    print(f"Error: {stderr[:300]}")
if not shutil.which("k"):
    print("Kona not installed. Expected output:")
    print("XOR predictions:")
    print("0 0.0 -> ,0.003721189")
    print("0 1.0 -> ,0.9952888")
    print("1 0.0 -> ,0.9952321")
    print("1 1.0 -> ,0.006605917")

## What we learned

1. **Claude can generate valid code in extremely niche languages.** K/Kona has minimal training data, yet Claude produces working array programs with correct use of adverbs, implicit arguments, and K-specific idioms.

2. **Self-repair works for terse languages.** When Claude's first attempt has a bug, feeding the Kona error back lets it converge on working code — often in 1-2 iterations.

3. **Array thinking is fundamentally different.** There are no `for` loops in K. Everything is expressed as operations over entire arrays — `_dot\:` replaces nested loops for matmul, `*\:` computes outer products in one expression. Claude correctly maps neural net operations onto these primitives.

4. **The entire neural net fits in ~30 lines.** Forward pass, backprop, 20000-epoch training loop, and inference. The equivalent PyTorch code would be 50-80 lines.

### Try it yourself

- Change the architecture: more hidden units, or add a second hidden layer
- Try a different activation: replace `sig` with `_tanh` (adjust the derivative!)
- Ask Claude to write it in other array languages: APL, J, or Q/KDB+
- Swap in softmax + cross-entropy for multi-class classification